In [8]:
import pandas as pd

# 0.Processing

In [2]:
gene_group_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/04.virtual_knockout/gene_groups_matrix_kwx.csv"
gene_score_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/gene_score_pr_activate_kwx.csv"   # 无表头：第1列gene，第2列score
out_dir = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/"

# ========= 1) 读 gene_groups_matrix =========
groups = pd.read_csv(gene_group_path)

inc_genes = (
    groups["increase_coupling_genes"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

dec_genes = (
    groups["decrease_coupling_genes"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Increase coupling genes:", len(inc_genes))
print("Decrease coupling genes:", len(dec_genes))

# ========= 2) 读 gene_score（无表头） =========
gene_score = pd.read_csv(gene_score_path, header=None, names=["gene", "score"])
gene_score["gene"] = gene_score["gene"].astype(str).str.strip()

print("Total rows in gene_score:", gene_score.shape[0])

Increase coupling genes: 636
Decrease coupling genes: 927
Total rows in gene_score: 15634


# 1.Match Gene

In [3]:
# ========= 3) 用基因名去匹配 score =========
inc_scores = gene_score[gene_score["gene"].isin(inc_genes)].copy()
dec_scores = gene_score[gene_score["gene"].isin(dec_genes)].copy()

print("Matched increase genes with score:", inc_scores.shape[0])
print("Matched decrease genes with score:", dec_scores.shape[0])

# （可选）如果 gene_score 里有重复基因名，保留分数更大的那条；不需要就删掉这两行
# inc_scores = inc_scores.sort_values("score", ascending=False).drop_duplicates("gene")
# dec_scores = dec_scores.sort_values("score", ascending=False).drop_duplicates("gene")

Matched increase genes with score: 636
Matched decrease genes with score: 927


# 2.Save

In [5]:
# ========= 4) 保存两个文件 =========
inc_scores.to_csv(out_dir + "gene_score_activate_increase_coupling_kwx.csv", index=False)
dec_scores.to_csv(out_dir + "gene_score_activate_decrease_coupling_kwx.csv", index=False)

print("Saved:")
print(" -", out_dir + "gene_score_activate_increase_coupling_kwx.csv")
print(" -", out_dir + "gene_score_activate_decrease_coupling_kwx.csv")

Saved:
 - /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/gene_score_activate_increase_coupling_kwx.csv
 - /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/gene_score_activate_decrease_coupling_kwx.csv


# 3.Plot

In [11]:
gene_score = pd.read_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/gene_score_pr_activate_kwx.csv", header=None, index_col=0)
expr = pd.read_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_Schaefer2018_400Parcels_centroids_kwx.csv", index_col=0)

all_genes = gene_score.index
common_genes = expr.columns.intersection(all_genes)
expr_all_genes = expr.loc[:, common_genes]
print(f"gene_score 中基因数: {len(all_genes)}")
print(f"表达矩阵中匹配到的基因数: {len(common_genes)}")
print(f"匹配率: {len(common_genes) / len(all_genes):.2%}")
expr_all_genes.to_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/expr_left_all_activate_Schaefer2018_400Parcels_kwx.csv")
all_scores = gene_score.loc[common_genes, 1]
weighted_region_scores = expr_all_genes.dot(all_scores)
weighted_region_scores.to_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/weighted_region_scores_activate_all_Schaefer2018_400Parcels_kwx.csv",header=False)

gene_score 中基因数: 15634
表达矩阵中匹配到的基因数: 15631
匹配率: 99.98%


# 4.脑区基因表达平均值（无基因得分介入）

In [2]:
import pandas as pd

# 读取 CSV 文件，第一列是脑区名称，第一行是基因名称
file_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/expr_left_increase_activate_kwx.csv"  # 替换为你的文件路径
df = pd.read_csv(file_path, index_col=0)

# 计算每个脑区的基因表达值的平均值
mean_expression = df.mean(axis=1)

# 将结果保存到新的 CSV 文件
mean_expression.to_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/05.new_gene_score_list/mean_region_scores_increase_kwx.csv", header=True)

# 输出结果
print("每个脑区的基因表达值的平均值：")
print(mean_expression)

每个脑区的基因表达值的平均值：
lh.lateralorbitofrontal_1    0.489521
lh.lateralorbitofrontal_2    0.523538
lh.parsorbitalis_1           0.523672
lh.frontalpole_1             0.412809
lh.medialorbitofrontal_1     0.489290
                               ...   
Left-Putamen                 0.473967
Left-Pallidum                0.486784
Left-Accumbens-area          0.479948
Left-Hippocampus             0.482973
Left-Amygdala                0.465072
Length: 64, dtype: float64
